# Training CNN sehat/sakit di Google Colab (v4: + kerusakan hama + daun tropis umum)

**Sebelum mulai**
1. Lokal: `python scripts/make_colab_bundle.py` → upload `colab_bundle.zip` ke **Google Drive → MyDrive/** (timpa yang lama).
2. Colab: **Runtime → Change runtime type → T4 GPU**, lalu **Runtime → Disconnect and delete runtime**.
3. **Runtime → Run all** (sel TEST otomatis dilewati). Pastikan sel 1 menampilkan GPU.
4. Kirim output sel 5 (VAL) ke Claude. Setelah disetujui: `RUN_TEST = True` di sel 7, jalankan sel 7 & 8.

## 1. Setup

In [ ]:
!nvidia-smi -L
# versi sama dengan lokal agar models/best.keras bisa dibuka di laptop; ai-edge-litert = interpreter TFLite (deploy)
!pip -q install tensorflow==2.20.0 keras==3.10.0 ai-edge-litert
!python -c "import tensorflow as tf, keras; print('TF', tf.__version__, '| Keras', keras.__version__, '| GPU', tf.config.list_physical_devices('GPU'))"

## 2. Data dari Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/proj && mkdir -p /content/proj
!cp /content/drive/MyDrive/colab_bundle.zip /content/ && unzip -q -o /content/colab_bundle.zip -d /content/proj
%cd /content/proj

import csv, glob
from collections import Counter
rows = list(csv.DictReader(open('data/processed/split.csv', encoding='utf-8')))
n_jpg = len(glob.glob('data/processed/*/*/*.jpg'))
assert n_jpg == len(rows), f'jumlah citra {n_jpg} != split.csv {len(rows)}'
print('citra:', n_jpg)  # v2: 27812
print('per split x sumber:', dict(sorted(Counter((r['split'], r['source']) for r in rows).items())))

## 3. Cek arsitektur

In [ ]:
!python model.py 2>&1 | tail -3

## 4. Training v4 (SATU run, config R1: tanpa class_weight, patience 12): ±75-90 menit di T4
Train = data v3 + iNaturalist v4 (kerusakan hama + daun tropis umum) + `bgaug`. Val/test ikut berisi data baru.

In [ ]:
RUNS = {'V4': '--no-class-weight --patience 12'}
for name, args in RUNS.items():
    print(f'
########## {name}: {args}')
    !rm -rf models && python train.py {args} 2>&1 | grep -E 'args:|GPU:|Found|val_accuracy|Restoring|early stopping|Model terbaik'
    !python evaluate.py --split val > /dev/null 2>&1 && rm -rf models_{name} && cp -r models models_{name}
    # simpan LANGSUNG ke Drive: runtime Colab bisa ter-reset saat idle dan menghapus /content
    !rm -rf /content/drive/MyDrive/models_{name} && cp -r models_{name} /content/drive/MyDrive/
print('selesai, tersimpan juga di Drive: MyDrive/models_*')


## 5. Ringkasan VAL v4 (kirim output sel ini ke Claude)

In [ ]:
%cd /content
import glob, os
from IPython.display import Image, display
# pulihkan dari Drive jika runtime sempat ter-reset (proyek di-unzip ulang dulu bila perlu)
if not os.path.isdir('/content/proj'):
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/proj && cp /content/drive/MyDrive/colab_bundle.zip /content/ && unzip -q -o /content/colab_bundle.zip -d /content/proj
%cd /content/proj
for d in glob.glob('/content/drive/MyDrive/models_*'):
    if os.path.isdir(d) and not os.path.isdir(os.path.basename(d)):
        !cp -r "{d}" .
runs = sorted(d for d in glob.glob('models_*') if os.path.isdir(d))
assert runs, 'models_* tidak ada di runtime maupun Drive: jalankan ulang sel 2 & 4'
for d in runs:
    print(f'
########## {d}')
    !grep -E '^==|true|accuracy' {d}/eval_val/report.txt
    display(Image(f'{d}/history.png'))


## 6. Ekspor TFLite v4

In [ ]:
%cd /content/proj
from IPython.display import Image, display
BEST = 'V4'
!rm -rf models && cp -r models_{BEST} models && python inference.py --export 2>&1 | tail -1

## 7. Evaluasi TEST: SEKALI saja, setelah VAL disetujui (angka untuk jurnal)

In [ ]:
%cd /content/proj
from IPython.display import Image, display
RUN_TEST = False  # ubah ke True setelah hasil VAL disetujui

if RUN_TEST:
    !python evaluate.py --split test 2>&1 | sed -n '/== KESELURUHAN/,$p'
    display(Image('models/eval_test/confusion_matrix.png'))
else:
    print('Dilewati: set RUN_TEST = True setelah VAL disetujui.')

## 8. Simpan hasil ke Drive + unduh `models_v4.zip`

In [ ]:
%cd /content/proj
!cd models && zip -q -r /content/models_v4.zip .
!cp /content/models_v4.zip /content/drive/MyDrive/
!unzip -l /content/models_v4.zip | grep -E 'best.keras|model.tflite|report.txt'
from google.colab import files
files.download('/content/models_v4.zip')